## Authenticate to GCP

In [1]:
import os
import json
from google.colab import userdata

# Write service account credentials to a temp file
sa_json = userdata.get('GCP_SERVICE_ACCOUNT')
sa_path = '/tmp/gcp_sa.json'
with open(sa_path, 'w') as f:
    f.write(sa_json)
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = sa_path

# Also authenticate gsutil and gcloud CLI
!gcloud auth activate-service-account --key-file=/tmp/gcp_sa.json --quiet

# Set project (replace with your actual project ID)
PROJECT_ID = 'recosys-489001'  # <-- CHANGE THIS to your actual GCP project ID
!gcloud config set project {PROJECT_ID} --quiet

# Verify connection
from google.cloud import storage
client = storage.Client(project=PROJECT_ID)

BUCKET_NAME = 'recosys-data-bucket'  # <-- CHANGE THIS to your actual bucket name
bucket = client.bucket(BUCKET_NAME)
print(f"✅ Connected to bucket: {bucket.name}")

Activated service account credentials for: [921967012784-compute@developer.gserviceaccount.com]


To take a quick anonymous survey, run:
  $ gcloud survey

- '@type': type.googleapis.com/google.rpc.ErrorInfo
  domain: googleapis.com
  metadata:
    activationUrl: https://console.developers.google.com/apis/api/cloudresourcemanager.googleapis.com/overview?project=921967012784
    consumer: projects/921967012784
    containerInfo: '921967012784'
    service: cloudresourcemanager.googleapis.com
    serviceTitle: Cloud Resource Manager API
  reason: SERVICE_DISABLED
- '@type': type.googleapis.com/google.rpc.LocalizedMessage
  locale: en-US
  message: Cloud Resource Manager API has not been used in project 921967012784 before
    or it is disabled. Enable it by visiting https://console.developers.google.com/apis/api/cloudresourcemanager.googleapis.com/overview?project=921967012784
    then retry. If you enabled this API recently, wait a few minutes for the action
    to propagate to our syst

## Download Dataset from Kaggle

In [2]:
'''# Set Kaggle credentials
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Install Kaggle CLI
!pip install kaggle --quiet

# Download the dataset (this downloads to Colab's temporary disk)
# The -p flag specifies the download directory
!kaggle datasets download \
    -d mkechinov/ecommerce-behavior-data-from-multi-category-store \
    -p /tmp/kaggle_data/

print("\n✅ Download complete. Contents:")
!ls -lh /tmp/kaggle_data/'''

'# Set Kaggle credentials\nos.environ[\'KAGGLE_USERNAME\'] = userdata.get(\'KAGGLE_USERNAME\')\nos.environ[\'KAGGLE_KEY\'] = userdata.get(\'KAGGLE_KEY\')\n\n# Install Kaggle CLI\n!pip install kaggle --quiet\n\n# Download the dataset (this downloads to Colab\'s temporary disk)\n# The -p flag specifies the download directory\n!kaggle datasets download     -d mkechinov/ecommerce-behavior-data-from-multi-category-store     -p /tmp/kaggle_data/\n\nprint("\n✅ Download complete. Contents:")\n!ls -lh /tmp/kaggle_data/'

In [3]:
'''!mkdir -p /tmp/raw_data
!unzip -o '/tmp/kaggle_data/*.zip' -d /tmp/raw_data/

print("\n✅ Unzipped files:")
!ls -lh /tmp/raw_data/'''

'!mkdir -p /tmp/raw_data\n!unzip -o \'/tmp/kaggle_data/*.zip\' -d /tmp/raw_data/\n\nprint("\n✅ Unzipped files:")\n!ls -lh /tmp/raw_data/'

In [4]:
# ============================================================
# DOWNLOAD ALL 7 MONTHS OF DATA
# ============================================================
# STEP 1: Paste the actual download URLs from the Kaggle page
# Right-click each link → Copy Link Address → paste below
# ============================================================

files = {
    "2019-Oct.csv.gz": "https://data.rees46.com/datasets/marketplace/2019-Oct.csv.gz",
    "2019-Nov.csv.gz": "https://data.rees46.com/datasets/marketplace/2019-Nov.csv.gz",
    "2019-Dec.csv.gz": "https://data.rees46.com/datasets/marketplace/2019-Dec.csv.gz",
    "2020-Jan.csv.gz": "https://data.rees46.com/datasets/marketplace/2020-Jan.csv.gz",
    "2020-Feb.csv.gz": "https://data.rees46.com/datasets/marketplace/2020-Feb.csv.gz",
    "2020-Mar.csv.gz": "https://data.rees46.com/datasets/marketplace/2020-Mar.csv.gz",
    "2020-Apr.csv.gz": "https://data.rees46.com/datasets/marketplace/2020-Apr.csv.gz",
}

!pip install gdown --quiet
import gdown
import os

!mkdir -p /tmp/raw_data

for filename, url in files.items():
    csv_name = filename.replace('.gz', '')
    print(f"\n{'='*60}")
    print(f"Processing {filename}...")

    # --- DOWNLOAD ---
    # Try gdown first (handles Google Drive links)
    # Falls back to wget for direct URLs
    dest = f"/tmp/raw_data/{filename}"

    if 'drive.google.com' in url or 'docs.google.com' in url:
        # Google Drive link — extract file ID and use gdown
        gdown.download(url, dest, quiet=False, fuzzy=True)
    elif 'dropbox.com' in url:
        # Dropbox link — replace dl=0 with dl=1 for direct download
        direct_url = url.replace('dl=0', 'dl=1')
        !wget -q --show-progress -O {dest} "{direct_url}"
    else:
        # Generic direct URL
        !wget -q --show-progress -O {dest} "{url}"

    # Verify download
    if not os.path.exists(dest) or os.path.getsize(dest) < 1_000_000:
        print(f"❌ FAILED: {filename} — check the URL")
        continue

    print(f"  Downloaded: {os.path.getsize(dest) / 1e9:.2f} GB (compressed)")

    # --- DECOMPRESS ---
    print(f"  Decompressing...")
    !gunzip -f {dest}

    csv_path = f"/tmp/raw_data/{csv_name}"
    print(f"  Decompressed: {os.path.getsize(csv_path) / 1e9:.2f} GB")

    # --- UPLOAD TO GCS ---
    print(f"  Uploading to GCS...")
    !gsutil cp {csv_path} gs://{BUCKET_NAME}/raw/

    # --- FREE DISK SPACE ---
    os.remove(csv_path)
    print(f"  ✅ {csv_name} → GCS done, local copy removed")

# ============================================================
# FINAL VERIFICATION
# ============================================================
print(f"\n{'='*60}")
print("All files in GCS:")
!gsutil ls -lh gs://{BUCKET_NAME}/raw/


Processing 2019-Oct.csv.gz...
/tmp/raw_data/2019- 100%[===================>]   1.62G  23.9MB/s    in 71s     
  Downloaded: 1.74 GB (compressed)
  Decompressing...
  Decompressed: 5.67 GB
  Uploading to GCS...
Copying file:///tmp/raw_data/2019-Oct.csv [Content-Type=text/csv]...
==> NOTE: You are uploading one or more large file(s), which would run
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables downloads of composite objects.

- [1 files][  5.3 GiB/  5.3 GiB]   78.1 MiB/

## Upload to GCS Bucket:

In [5]:
'''# Upload all CSVs to the raw/ folder in GCS
# -m flag enables parallel upload (much faster for large files)
!gsutil -m cp /tmp/raw_data/*.csv gs://{BUCKET_NAME}/raw/

print("\n✅ Upload complete. Files in GCS:")
!gsutil ls -lh gs://{BUCKET_NAME}/raw/'''

'# Upload all CSVs to the raw/ folder in GCS\n# -m flag enables parallel upload (much faster for large files)\n!gsutil -m cp /tmp/raw_data/*.csv gs://{BUCKET_NAME}/raw/\n\nprint("\n✅ Upload complete. Files in GCS:")\n!gsutil ls -lh gs://{BUCKET_NAME}/raw/'

In [6]:
# Peek at the first few rows directly from GCS (no full download needed)
print("Schema check — first 5 rows of 2019-Oct.csv:")
!gsutil cat -r 0-2000 gs://{BUCKET_NAME}/raw/2019-Oct.csv | head -5

print("\n\nSchema check — first 5 rows of 2019-Dec.csv (verify extra months match):")
!gsutil cat -r 0-2000 gs://{BUCKET_NAME}/raw/2019-Dec.csv | head -5

# Verify all 7 files are present
print("\n\nAll raw files:")
!gsutil ls -lh gs://{BUCKET_NAME}/raw/

Schema check — first 5 rows of 2019-Oct.csv:
event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713


Schema check — first 5 rows of 2019-Dec.csv (verify extra months match):
event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
2019-12-01 00:00:00 UTC,view,1005105,2232732093077520756,construction.tools.light,apple,1302.48,556695836,ca5eefc5-11f9-450c-91ed-380285a0bc80
2019

## Creating Samples for Initial Testing

In [2]:
!pip install polars --quiet
import polars as pl

# We'll work with October data for sampling (it's the smallest month)
# Download it temporarily
!gsutil cp gs://{BUCKET_NAME}/raw/2019-Oct.csv /tmp/oct_data.csv

print("Reading October data...")
df = pl.read_csv('/tmp/oct_data.csv')
print(f"October data shape: {df.shape}")
print(f"Columns: {df.columns}")
print(f"\nFirst 5 rows:")
print(df.head(5))

# Check basic stats
print(f"\nUnique users: {df.n_unique('user_id'):,}")
print(f"Unique products: {df.n_unique('product_id'):,}")
print(f"\nEvent distribution:")
print(df.group_by('event_type').len().sort('len', descending=True))
print(f"\nNull counts per column:")
print(df.null_count())

Copying gs://recosys-data-bucket/raw/2019-Oct.csv...
==> NOTE: You are downloading one or more large file(s), which would
run significantly faster if you enabled sliced object downloads. This
feature is enabled by default but requires that compiled crcmod be
installed (see "gsutil help crcmod").

-
Operation completed over 1 objects/5.3 GiB.                                      
Reading October data...
October data shape: (42448764, 9)
Columns: ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']

First 5 rows:
shape: (5, 9)
┌────────────┬────────────┬───────────┬───────────┬───┬──────────┬─────────┬───────────┬───────────┐
│ event_time ┆ event_type ┆ product_i ┆ category_ ┆ … ┆ brand    ┆ price   ┆ user_id   ┆ user_sess │
│ ---        ┆ ---        ┆ d         ┆ id        ┆   ┆ ---      ┆ ---     ┆ ---       ┆ ion       │
│ str        ┆ str        ┆ ---       ┆ ---       ┆   ┆ str      ┆ f64     ┆ i64       ┆ ---       

In [3]:
df.null_count()

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,13515609,6113008,0,0,2


In [4]:
# --- CREATE 50K SAMPLE ---
# Strategy: sample USERS who have at least 5 events, take ALL their events

# Count events per user
user_counts = df.group_by('user_id').agg(pl.len().alias('event_count'))

# Filter to users with at least 5 events (meaningful history)
eligible_users = user_counts.filter(pl.col('event_count') >= 5)
print(f"Users with >= 5 events: {eligible_users.height:,} out of {user_counts.height:,}")

# For 50K sample: aim for ~50K rows by sampling enough users
# Average events per eligible user:
avg_events = eligible_users.select(pl.col('event_count').mean()).item()
print(f"Average events per eligible user: {avg_events:.1f}")

# Calculate how many users we need for ~50K rows
users_needed_50k = int(50_000 / avg_events)
print(f"Sampling ~{users_needed_50k} users for 50K-row sample")

# Sample users
sampled_users_50k = eligible_users.sample(n=min(users_needed_50k, eligible_users.height), seed=42)

# Get ALL events for sampled users
sample_50k = df.join(sampled_users_50k.select('user_id'), on='user_id', how='inner')
print(f"\n50K Sample: {sample_50k.height:,} rows, {sample_50k.n_unique('user_id'):,} users, {sample_50k.n_unique('product_id'):,} products")

# Verify: check event distribution is similar to original
print("\n50K Sample event distribution:")
print(sample_50k.group_by('event_type').len().sort('len', descending=True))

Users with >= 5 events: 1,503,009 out of 3,022,290
Average events per eligible user: 26.2
Sampling ~1905 users for 50K-row sample

50K Sample: 50,192 rows, 1,905 users, 13,340 products

50K Sample event distribution:
shape: (3, 2)
┌────────────┬───────┐
│ event_type ┆ len   │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ view       ┆ 47940 │
│ cart       ┆ 1294  │
│ purchase   ┆ 958   │
└────────────┴───────┘


In [5]:
# --- CREATE 500K SAMPLE ---
users_needed_500k = int(500_000 / avg_events)
print(f"Sampling ~{users_needed_500k} users for 500K-row sample")

sampled_users_500k = eligible_users.sample(
    n=min(users_needed_500k, eligible_users.height), seed=42
)
sample_500k = df.join(sampled_users_500k.select('user_id'), on='user_id', how='inner')
print(f"\n500K Sample: {sample_500k.height:,} rows, "
      f"{sample_500k.n_unique('user_id'):,} users, "
      f"{sample_500k.n_unique('product_id'):,} products")

print("\n500K Sample event distribution:")
print(sample_500k.group_by('event_type').len().sort('len', descending=True))

Sampling ~19053 users for 500K-row sample

500K Sample: 502,920 rows, 19,053 users, 53,332 products

500K Sample event distribution:
shape: (3, 2)
┌────────────┬────────┐
│ event_type ┆ len    │
│ ---        ┆ ---    │
│ str        ┆ u32    │
╞════════════╪════════╡
│ view       ┆ 482709 │
│ cart       ┆ 11364  │
│ purchase   ┆ 8847   │
└────────────┴────────┘


In [6]:
# --- VALIDATE SAMPLE QUALITY ---
# The key check: does the sample preserve the statistical properties of the full data?

print("=" * 60)
print("SAMPLE QUALITY VALIDATION")
print("=" * 60)

for name, s in [("Full Oct", df), ("50K Sample", sample_50k), ("500K Sample", sample_500k)]:
    print(f"\n--- {name} ---")
    total = s.height
    purchases = s.filter(pl.col('event_type') == 'purchase').height
    carts = s.filter(pl.col('event_type') == 'cart').height
    views = s.filter(pl.col('event_type') == 'view').height
    print(f"  Rows: {total:,}")
    print(f"  Users: {s.n_unique('user_id'):,}")
    print(f"  Products: {s.n_unique('product_id'):,}")
    print(f"  View %:     {views/total*100:.1f}%")
    print(f"  Cart %:     {carts/total*100:.1f}%")
    print(f"  Purchase %: {purchases/total*100:.1f}%")
    print(f"  Avg events/user: {total / s.n_unique('user_id'):.1f}")
    null_cat = s.select(pl.col('category_code').is_null().mean()).item()
    null_brand = s.select(pl.col('brand').is_null().mean()).item()
    print(f"  Null category_code: {null_cat*100:.1f}%")
    print(f"  Null brand: {null_brand*100:.1f}%")

# The percentages should be roughly similar across all three
# If they're wildly different, the sampling is biased

SAMPLE QUALITY VALIDATION

--- Full Oct ---
  Rows: 42,448,764
  Users: 3,022,290
  Products: 166,794
  View %:     96.1%
  Cart %:     2.2%
  Purchase %: 1.7%
  Avg events/user: 14.0
  Null category_code: 31.8%
  Null brand: 14.4%

--- 50K Sample ---
  Rows: 50,192
  Users: 1,905
  Products: 13,340
  View %:     95.5%
  Cart %:     2.6%
  Purchase %: 1.9%
  Avg events/user: 26.3
  Null category_code: 33.1%
  Null brand: 15.7%

--- 500K Sample ---
  Rows: 502,920
  Users: 19,053
  Products: 53,332
  View %:     96.0%
  Cart %:     2.3%
  Purchase %: 1.8%
  Avg events/user: 26.4
  Null category_code: 31.7%
  Null brand: 14.7%


In [7]:
# --- SAVE SAMPLES TO GCS ---
sample_50k.write_parquet('/tmp/events_50k_users.parquet')
sample_500k.write_parquet('/tmp/events_500k_users.parquet')

!gsutil cp /tmp/events_50k_users.parquet gs://{BUCKET_NAME}/samples/
!gsutil cp /tmp/events_500k_users.parquet gs://{BUCKET_NAME}/samples/

print("\n✅ Samples uploaded to GCS:")
!gsutil ls -lh gs://{BUCKET_NAME}/samples/

# Clean up local files to free disk
!rm /tmp/oct_data.csv /tmp/events_50k_users.parquet /tmp/events_500k_users.parquet

Copying file:///tmp/events_50k_users.parquet [Content-Type=application/octet-stream]...
\
Operation completed over 1 objects/866.9 KiB.                                    
Copying file:///tmp/events_500k_users.parquet [Content-Type=application/octet-stream]...
/
Operation completed over 1 objects/8.5 MiB.                                      

✅ Samples uploaded to GCS:
       0 B  2026-03-02T22:44:10Z  gs://recosys-data-bucket/samples/
  8.47 MiB  2026-03-03T01:15:49Z  gs://recosys-data-bucket/samples/events_500k_users.parquet
 866.9 KiB  2026-03-03T01:15:44Z  gs://recosys-data-bucket/samples/events_50k_users.parquet
TOTAL: 3 objects, 9769201 bytes (9.32 MiB)


## Final Verification of Everything in GCS

In [12]:
print("=" * 60)
print("FINAL GCS BUCKET STATUS")
print("=" * 60)

print("\n📁 raw/")
!gsutil ls -lh gs://{BUCKET_NAME}/raw/

print("\n📁 samples/")
!gsutil ls -lh gs://{BUCKET_NAME}/samples/

print("\n📁 All top-level folders:")
!gsutil ls gs://{BUCKET_NAME}/

print("\n✅ Setup complete!")
print(f"\nBucket name for teammates: {BUCKET_NAME}")
print(f"Project ID for teammates: {PROJECT_ID}")

FINAL GCS BUCKET STATUS

📁 raw/
       0 B  2026-03-02T22:44:01Z  gs://recosys-data-bucket/raw/
  8.72 GiB  2026-03-03T00:17:04Z  gs://recosys-data-bucket/raw/2019-Dec.csv
  8.39 GiB  2026-03-03T00:11:22Z  gs://recosys-data-bucket/raw/2019-Nov.csv
  5.28 GiB  2026-03-03T00:06:01Z  gs://recosys-data-bucket/raw/2019-Oct.csv
  8.63 GiB  2026-03-03T00:36:47Z  gs://recosys-data-bucket/raw/2020-Apr.csv
  7.15 GiB  2026-03-03T00:26:12Z  gs://recosys-data-bucket/raw/2020-Feb.csv
  7.24 GiB  2026-03-03T00:21:35Z  gs://recosys-data-bucket/raw/2020-Jan.csv
  7.29 GiB  2026-03-03T00:30:41Z  gs://recosys-data-bucket/raw/2020-Mar.csv
TOTAL: 8 objects, 56577978747 bytes (52.69 GiB)

📁 samples/
       0 B  2026-03-02T22:44:10Z  gs://recosys-data-bucket/samples/
  8.47 MiB  2026-03-03T01:15:49Z  gs://recosys-data-bucket/samples/events_500k_users.parquet
 866.9 KiB  2026-03-03T01:15:44Z  gs://recosys-data-bucket/samples/events_50k_users.parquet
TOTAL: 3 objects, 9769201 bytes (9.32 MiB)

📁 All top-level